In [1]:
from pybaseball import statcast_batter
import pandas as pd

# Pull all pitch-level Statcast data for Bobby Witt Jr. (player_id=677951)
# from 2022 through end of 2025 season.
# Returns one row per pitch — 10,689 pitches across ~620 games, 118 columns.
# Key columns: launch_speed (exit velo), launch_angle, launch_speed_angle (barrel classification),
# and events (what happened: home_run, single, field_out, etc.)

df = statcast_batter('2022-04-01', '2025-10-01', player_id=677951)
print(df.shape)
print(df[['game_date', 'launch_speed', 'launch_angle', 'launch_speed_angle', 'events']].head(20))

Gathering Player Data
(10689, 118)
     game_date  launch_speed  launch_angle  launch_speed_angle     events
0   2025-09-28         101.4         -19.0                 2.0  field_out
1   2025-09-28           NaN           NaN                 NaN        NaN
2   2025-09-28         104.7          18.0                 5.0     double
3   2025-09-28           NaN           NaN                 NaN        NaN
4   2025-09-28          96.8          20.0                 4.0  field_out
5   2025-09-28           NaN           NaN                 NaN        NaN
6   2025-09-28           NaN           NaN                 NaN        NaN
7   2025-09-28           NaN           NaN                 NaN        NaN
8   2025-09-28          68.6          10.0                 NaN        NaN
9   2025-09-28          84.0          59.0                 3.0  field_out
10  2025-09-28         108.8          -1.0                 4.0  field_out
11  2025-09-28          75.1          15.0                 NaN        NaN
12 

In [2]:
# Filter to batted balls only — rows where launch_speed is NaN are 
# pitches not put in play (strikeouts, walks, fouls), we don't want those
batted = df[df['launch_speed'].notna()].copy()

# Aggregate to game level — one row per game
# This collapses 10,689 pitch rows down to one row per game date
game_stats = batted.groupby('game_date').agg(
    # Average exit velocity across all batted balls that game
    avg_exit_velo=('launch_speed', 'mean'),
    # Hardest hit ball of the game
    max_exit_velo=('launch_speed', 'max'),
    # Barrels = launch_speed_angle of 6, the best possible contact category
    barrel_count=('launch_speed_angle', lambda x: (x == 6).sum()),
    # Hard hit = exit velocity 95mph or higher
    hard_hit_count=('launch_speed', lambda x: (x >= 95).sum()),
    # Total batted balls that game (denominator for rates)
    batted_balls=('launch_speed', 'count'),
).reset_index()

# Barrel rate = barrels / total batted balls
# Hard hit rate = hard hit balls / total batted balls
game_stats['barrel_rate'] = game_stats['barrel_count'] / game_stats['batted_balls']
game_stats['hard_hit_rate'] = game_stats['hard_hit_count'] / game_stats['batted_balls']

print(game_stats.shape)
print(game_stats.head(10))

(641, 8)
    game_date  avg_exit_velo  max_exit_velo  barrel_count  hard_hit_count  \
0  2022-04-07      91.575000          110.4             0               1   
1  2022-04-09      79.837500           96.5             0               2   
2  2022-04-10      77.350000          110.1             0               1   
3  2022-04-11      83.700000          103.3             0               1   
4  2022-04-12      84.833333           97.1             0               1   
5  2022-04-14      85.000000          105.6             0               1   
6  2022-04-15      85.316667          104.0             0               2   
7  2022-04-16      76.585714           92.7             0               0   
8  2022-04-19      86.240000          104.0             0               2   
9  2022-04-20      90.566667          100.4             0               1   

   batted_balls  barrel_rate  hard_hit_rate  
0             4          0.0       0.250000  
1             8          0.0       0.250000  
2    

In [3]:
# Sort chronologically
game_stats = game_stats.sort_values('game_date').reset_index(drop=True)

# Rolling averages — shift(1) so we never leak current game into the feature
game_stats['avg_exit_velo_7']   = game_stats['avg_exit_velo'].shift(1).rolling(7,  min_periods=3).mean()
game_stats['barrel_rate_7']     = game_stats['barrel_rate'].shift(1).rolling(7,  min_periods=3).mean()
game_stats['hard_hit_rate_7']   = game_stats['hard_hit_rate'].shift(1).rolling(7,  min_periods=3).mean()
game_stats['avg_exit_velo_15']  = game_stats['avg_exit_velo'].shift(1).rolling(15, min_periods=7).mean()
game_stats['barrel_rate_15']    = game_stats['barrel_rate'].shift(1).rolling(15, min_periods=7).mean()

print(game_stats[['game_date', 'avg_exit_velo', 'avg_exit_velo_7', 'barrel_rate_7', 'hard_hit_rate_7']].head(15))

     game_date  avg_exit_velo  avg_exit_velo_7  barrel_rate_7  hard_hit_rate_7
0   2022-04-07      91.575000              NaN            NaN              NaN
1   2022-04-09      79.837500              NaN            NaN              NaN
2   2022-04-10      77.350000              NaN            NaN              NaN
3   2022-04-11      83.700000        82.920833            0.0         0.200000
4   2022-04-12      84.833333        83.115625            0.0         0.212500
5   2022-04-14      85.000000        83.459167            0.0         0.236667
6   2022-04-15      85.316667        83.715972            0.0         0.252778
7   2022-04-16      76.585714        83.944643            0.0         0.264286
8   2022-04-19      86.240000        81.803316            0.0         0.228571
9   2022-04-20      90.566667        82.717959            0.0         0.250000
10  2022-04-21      79.060000        84.606054            0.0         0.283333
11  2022-04-22      88.200000        83.943197      